# Can Ouro tell it has been steered?

Texts are generated from the same prompts under several conditions: natural, sentiment-steered (both signs, CAA at loop 4 after 12 layers, 20% of residual norm), steered with unrelated concepts (past tense, animals), and content-matched controls where the model is merely *asked* for a negative or positive tone. Every text is then teacher-forced through the unsteered model, and we look for a "this is not what I would have written" signal in the activations and in the model's own surprise, separated from "this text is negative / about restaurants" by three debiasing tests: transfer to unseen concepts, projecting the concept directions out, and steered-positive vs natural (same sentiment).

Model: `ByteDance/Ouro-1.4B-Thinking`, fp16, transformers 4.54.1 with the cache patch.

In [1]:
%pip install -q "transformers==4.54.1" accelerate huggingface_hub
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.54.1 | torch 2.11.0+cu128 | cuda True


In [2]:
import sys, torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

model_name = "ByteDance/Ouro-1.4B-Thinking"
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, config=config, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)

# transformers>=4.54 made Cache.key_cache/value_cache read-only properties; Ouro's cache assigns to them.
def _patch_ouro_cache(model):
    cls = sys.modules[type(model).__module__].UniversalTransformerCache
    for name in ("key_cache", "value_cache"):
        priv = "_ouro_" + name
        setattr(cls, name, property(lambda self, priv=priv: self.__dict__.setdefault(priv, []),
                                    lambda self, v, priv=priv: self.__dict__.__setitem__(priv, v)))
_patch_ouro_cache(model)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
H, L, S = config.hidden_size, config.num_hidden_layers, config.total_ut_steps
layers = model.model.layers
print("loaded on", model.device, "| hidden", H, "| layers", L, "| loops", S)

config.json: 0.00B [00:00, ?B/s]

configuration_ouro.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ByteDance/Ouro-1.4B-Thinking:
- configuration_ouro.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

modeling_ouro.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ByteDance/Ouro-1.4B-Thinking:
- modeling_ouro.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/2.87G [00:00<?, ?B/s]

loaded on cuda:0 | hidden 2048 | layers 24 | loops 4


In [3]:
# ---------- Steering toolkit: CAA vectors after 12 layers, per loop, for three concepts ----------
import numpy as np, torch, json, time
IDX = 11   # decoder layer index whose output is "after 12 layers"

CTX = ["yesterday", "at the park", "in the garden", "near the river", "at school", "in town", "on the road", "at the market",
       "by the lake", "in the city", "on the farm", "at the beach", "in the forest", "at home", "in the shop", "on the hill"]
SUBJ = ["the dog", "my sister", "the teacher", "our neighbour", "the little boy", "a stranger", "the old man", "her friend",
        "the cat", "the driver", "the nurse", "the pilot", "a child", "the farmer", "my uncle", "the girl"]
NOUNS = ["movie", "meal", "hotel", "concert", "book", "service", "flight", "game", "lecture", "party", "trip", "museum", "show", "phone", "app", "coffee"]
CONCEPTS = {
  "sentiment": dict(extract=[t.format(n=n) for n in NOUNS for t in ("The {n} was", "I thought the {n} was", "In the end the {n} was")],
      pos=["wonderful", "great", "amazing", "excellent", "fantastic", "delightful", "lovely", "superb", "brilliant", "perfect", "good", "awesome"],
      neg=["terrible", "awful", "horrible", "disappointing", "dreadful", "bad", "poor", "boring", "miserable", "frustrating", "mediocre", "lousy"], sep=" "),
  "tense": dict(extract=[t.format(n=s) for s in SUBJ for t in ("{n}", "And then {n}", "As usual, {n}")],
      pos=["walked", "played", "jumped", "looked", "wanted", "stayed", "moved", "called", "asked", "liked", "worked", "opened"],
      neg=["walks", "plays", "jumps", "looks", "wants", "stays", "moves", "calls", "asks", "likes", "works", "opens"], sep=" "),
  "animal": dict(extract=[t.format(c=c) for c in CTX for t in ("{c}, I saw a", "{c}, we looked at a", "{c}, she pointed at a")],
      pos=["dog", "cat", "horse", "bird", "cow", "rabbit", "fox", "deer", "sheep", "goat", "wolf", "bear"],
      neg=["truck", "car", "bus", "train", "bike", "taxi", "tractor", "van", "boat", "plane", "jeep", "ship"], sep=" "),
}

def extract(C):
    sep = C["sep"]; ok = lambda w: len(tokenizer.encode(sep + w, add_special_tokens=False)) == 1
    pos, neg = [w for w in C["pos"] if ok(w)], [w for w in C["neg"] if ok(w)]; n = min(len(pos), len(neg), 8)
    texts, labels = [], []
    for pre in C["extract"]:
        for i in range(n): texts += [pre + sep + pos[i], pre + sep + neg[i]]; labels += [1, 0]
    sums = {s: torch.zeros(2, H, device=model.device) for s in range(S)}; nrm = {s: [0.0, 0] for s in range(S)}; cur = [None]
    def cap(mod, args, kwargs, out):
        h = (out[0] if isinstance(out, tuple) else out)[:, -1].float(); s = int(kwargs["current_ut"])
        for i, lab in enumerate(cur[0]): sums[s][lab] += h[i]
        nrm[s][0] += h.norm(dim=-1).sum().item(); nrm[s][1] += len(h)
    hc = layers[IDX].register_forward_hook(cap, with_kwargs=True)
    try:
        with torch.no_grad():
            for i in range(0, len(texts), 32):
                enc = tokenizer(texts[i:i+32], return_tensors="pt", padding=True).to(model.device); cur[0] = labels[i:i+32]
                model(**enc, use_cache=False)
    finally: hc.remove()
    n1 = sum(labels); n0 = len(labels) - n1
    v = {s: sums[s][1] / n1 - sums[s][0] / n0 for s in range(S)}
    return {s: v[s] / v[s].norm() for s in range(S)}, {s: nrm[s][0] / nrm[s][1] for s in range(S)}

UNIT, RESID = {}, {}
for name, C in CONCEPTS.items():
    UNIT[name], RESID[name] = extract(C)
    print(f"{name:<10} |h| per loop: " + " ".join(f"{RESID[name][s]:.1f}" for s in range(S)))

# permanent steering hook on layer 12; steer_cfg = {loop: vector} (empty = off)
steer_cfg = {}
def steer_hook(mod, args, kwargs, out):
    v = steer_cfg.get(int(kwargs["current_ut"]))
    if v is None: return None
    h = out[0] if isinstance(out, tuple) else out
    h2 = h + v.to(h.dtype)
    return (h2,) + tuple(out[1:]) if isinstance(out, tuple) else h2
steer_handle = layers[IDX].register_forward_hook(steer_hook, with_kwargs=True)
def set_steer(concept=None, alpha=0.0, loops=(3,)):
    steer_cfg.clear()
    if concept and alpha:
        for s in loops: steer_cfg[s] = alpha * RESID[concept][s] * UNIT[concept][s]
print("steering hook installed (loop-4 injection after 12 layers by default)")

sentiment  |h| per loop: 6.1 8.8 10.2 10.6
tense      |h| per loop: 5.4 7.4 8.4 9.0
animal     |h| per loop: 6.0 8.2 9.9 10.2
steering hook installed (loop-4 injection after 12 layers by default)


In [4]:
# ---------- Generate texts under each condition from the same prompts ----------
TOPICS = ["restaurant downtown", "hotel by the sea", "concert in the park", "science museum", "new cinema", "weekend market",
          "mountain hike", "city library", "football match", "art gallery", "train journey", "cooking class",
          "neighbourhood cafe", "botanical garden", "tech conference", "village fair"]
FRAMES = ["Let me tell you about the {t}. It", "Yesterday I spent the afternoon at the {t}. The", "Here are my thoughts on the {t}. Overall, it"]
PROMPTS = [f.format(t=t) for t in TOPICS for f in FRAMES]          # 48 prompts

CONDITIONS = {   # name: (prefix instruction for the prompt, steering concept, alpha)
    "natural":      ("", None, 0.0),
    "prompt_neg":   ("(Write this in a disappointed, negative tone.) ", None, 0.0),
    "prompt_pos":   ("(Write this in an enthusiastic, positive tone.) ", None, 0.0),
    "steer_neg":    ("", "sentiment", -0.2),
    "steer_pos":    ("", "sentiment", +0.2),
    "steer_tense":  ("", "tense", +0.2),
    "steer_animal": ("", "animal", +0.2),
}
ALPHA, NEW = 0.2, 60
texts = {}   # (condition, prompt_idx) -> continuation string
t0 = time.time()
for cname, (prefix, concept, alpha) in CONDITIONS.items():
    set_steer(concept, alpha)
    for i in range(0, len(PROMPTS), 16):
        batch = [prefix + p for p in PROMPTS[i:i+16]]
        enc = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
        torch.manual_seed(1000 + i)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=NEW, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
        for j, o in enumerate(out):
            texts[(cname, i + j)] = tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"  {cname:<13} done ({time.time()-t0:.0f}s)   e.g. {texts[(cname, 0)][:90]!r}")
set_steer()
json.dump({f"{c}|{i}": t for (c, i), t in texts.items()}, open("/content/steer_detect_texts.json", "w"), indent=1)
print(f"\n{len(texts)} texts saved")

  natural       done (27s)   e.g. "'s been open for only a month, but I've heard the food is amazing. My friend went there la"
  prompt_neg    done (56s)   e.g. '’s so crowded, and there’s no space to sit. I’m really frustrated because I wanted to have'
  prompt_pos    done (89s)   e.g. '’s a small, cozy place with a big heart. The owner is a passionate chef who takes pride in'
  steer_neg     done (116s)   e.g. "'s terrible. I can't stand the smell. It's the kind of smell you get from a sewer.\n\nWait, "
  steer_pos     done (143s)   e.g. '’s open until 11 PM, and the service is excellent. I’ll see you there!\n\nWhat is the key in'
  steer_tense   done (169s)   e.g. " was amazing! I loved the pizza, the pasta, and the desserts. I couldn't believe how fresh"
  steer_animal  done (195s)   e.g. "'s been open for only a few months, but I've been telling my friends about it ever since I"

336 texts saved


In [5]:
# ---------- Teacher-force every text through the UNSTEERED model; record activations and lens metrics per continuation token ----------
set_steer()
COND = list(CONDITIONS)
items = [(c, i) for c in COND for i in range(len(PROMPTS))]
full = [PROMPTS[i] + texts[(c, i)] for c, i in items]                 # scored WITHOUT any tone instruction
P_len = [len(tokenizer.encode(PROMPTS[i], add_special_tokens=False)) for c, i in items]

caps = {}; cnt = [0]
def cap_mid(mod, args, kwargs, out): caps[("mid", int(kwargs["current_ut"]))] = (out[0] if isinstance(out, tuple) else out).detach()
def cap_end(mod, inp, out): caps[("end", cnt[0])] = out.detach(); cnt[0] += 1
def cap_gate(mod, inp, out): caps[("gate", cnt[0] - 1)] = out.detach().float().squeeze(-1)
hooks = [layers[IDX].register_forward_hook(cap_mid, with_kwargs=True), model.model.norm.register_forward_hook(cap_end),
         model.model.early_exit_gate.register_forward_hook(cap_gate)]

meta = []                     # per continuation token
ACT = {f"mid{s+1}": [] for s in range(S)}; ACT["end4"] = []
t0 = time.time()
try:
    with torch.no_grad():
        for b in range(0, len(full), 16):
            enc = tokenizer(full[b:b+16], return_tensors="pt", padding=True).to(model.device)
            caps.clear(); cnt[0] = 0
            model(**enc, use_cache=False)
            ids, am = enc["input_ids"], enc["attention_mask"]
            logp = [torch.log_softmax(model.lm_head(caps[("end", s)]).float(), -1) for s in range(S)]   # S x [B, T, V]
            for j in range(ids.shape[0]):
                c, i = items[b + j]; T = int(am[j].sum()); start = ids.shape[1] - T; P = P_len[b + j]
                for t in range(start + P - 1, ids.shape[1] - 1):        # position t predicts token t+1 (a continuation token)
                    tgt = ids[j, t + 1]
                    lp4 = logp[3][j, t]; lp1 = logp[0][j, t]; lp3 = logp[2][j, t]
                    p4 = lp4.exp()
                    meta.append({"cond": c, "prompt": i, "pos": t + 1 - (start + P), "tok": tgt.item(),
                                 "nll": -lp4[tgt].item(), "ent": -(p4 * lp4).sum().item(),
                                 "kl14": (p4 * (lp4 - lp1)).sum().item(), "kl34": (p4 * (lp4 - lp3)).sum().item(),
                                 "gate1": torch.sigmoid(caps[("gate", 0)][j, t]).item(), "gate4": torch.sigmoid(caps[("gate", 3)][j, t]).item()})
                    for s in range(S): ACT[f"mid{s+1}"].append(caps[("mid", s)][j, t + 1].cpu())     # state AT the continuation token
                    ACT["end4"].append(caps[("end", 3)][j, t + 1].cpu())
            if b % 64 == 0: print(f"  {b+16}/{len(full)} texts, {len(meta):,} tokens, {time.time()-t0:.0f}s")
finally:
    for h in hooks: h.remove()
ACT = {k: torch.stack(v).to(torch.float16).numpy() for k, v in ACT.items()}
import pandas as pd
df = pd.DataFrame(meta)
print(f"\n{len(df):,} continuation tokens from {len(full)} texts; activation arrays: " + ", ".join(f"{k} {v.shape}" for k, v in ACT.items()))
print("\nPer-condition means over continuation tokens (model's own view of the text, unsteered):")
print(df.groupby("cond")[["nll", "ent", "kl14", "kl34", "gate1", "gate4"]].mean().round(3).loc[COND])

  16/336 texts, 937 tokens, 1s
  80/336 texts, 4,664 tokens, 6s
  144/336 texts, 8,504 tokens, 12s
  208/336 texts, 12,279 tokens, 17s
  272/336 texts, 16,101 tokens, 22s
  336/336 texts, 19,905 tokens, 27s

19,905 continuation tokens from 336 texts; activation arrays: mid1 (19905, 2048), mid2 (19905, 2048), mid3 (19905, 2048), mid4 (19905, 2048), end4 (19905, 2048)

Per-condition means over continuation tokens (model's own view of the text, unsteered):
                nll    ent   kl14   kl34  gate1  gate4
cond                                                  
natural       0.927  1.775  1.199  0.040  0.022  0.517
prompt_neg    1.197  1.972  1.385  0.036  0.017  0.520
prompt_pos    1.036  1.772  1.063  0.027  0.022  0.520
steer_neg     1.192  1.993  1.257  0.054  0.027  0.512
steer_pos     1.062  1.809  1.137  0.031  0.024  0.519
steer_tense   0.949  1.715  1.081  0.029  0.025  0.518
steer_animal  1.080  1.838  1.162  0.042  0.025  0.514


In [6]:
# ---------- Probes for "this text was steered", with debiasing ----------
import numpy as np, torch, json
rng = np.random.default_rng(0)
TEST_PROMPTS = set(i for i in range(len(PROMPTS)) if i % 4 == 0)      # 12 held-out prompts, 36 for training
is_test = df["prompt"].isin(TEST_PROMPTS).values
cond = df["cond"].values; prompt = df["prompt"].values

def auc(scores, y):
    order = np.argsort(scores); ranks = np.empty(len(scores)); ranks[order] = np.arange(1, len(scores) + 1)
    n1 = y.sum(); n0 = len(y) - n1
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else float("nan")

def text_level(scores, y, keys):
    """average token scores per text, then AUC/acc over texts"""
    d = {}
    for s_, y_, k in zip(scores, y, keys): d.setdefault(k, [[], y_])[0].append(s_)
    ts = np.array([np.mean(v[0]) for v in d.values()]); ty = np.array([v[1] for v in d.values()])
    return auc(ts, ty), float(((ts > 0.5) == ty).mean())

def fit_probe(X, y, epochs=8, lr=2e-3, wd=1e-3, bs=2048):
    dev = model.device
    X = torch.as_tensor(X); y = torch.as_tensor(y, dtype=torch.long)
    mu, sd = X.float().mean(0), X.float().std(0) + 1e-3
    lin = torch.nn.Linear(X.shape[1], 1).to(dev); opt = torch.optim.AdamW(lin.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(perm), bs):
            idx = perm[i:i+bs]; xb = ((X[idx].float() - mu) / sd).to(dev)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(lin(xb).squeeze(-1), y[idx].float().to(dev))
            opt.zero_grad(); loss.backward(); opt.step()
    def predict(Xn):
        with torch.no_grad():
            return torch.cat([torch.sigmoid(lin(((torch.as_tensor(Xn[i:i+8192]).float() - mu) / sd).to(dev)).squeeze(-1)).cpu()
                              for i in range(0, len(Xn), 8192)]).numpy()
    return predict

def project_out(X, feat):
    """remove the three concept directions (for the matching loop) from after-12 features"""
    if not feat.startswith("mid"): return X
    s = int(feat[3]) - 1
    Xf = torch.as_tensor(X).float()
    for cn in CONCEPTS:
        u = UNIT[cn][s].cpu(); Xf = Xf - (Xf @ u)[:, None] * u[None]
    return Xf.numpy()

def run(name, feat, pos_train, neg_train, tests, debias=False):
    """train pos vs neg on training prompts; evaluate on each test (pos_conds, neg_conds) on held-out prompts"""
    X = ACT[feat]; X = project_out(X, feat) if debias else X
    lab = np.isin(cond, pos_train).astype(int); use_tr = (~is_test) & np.isin(cond, pos_train + neg_train)
    predict = fit_probe(X[use_tr], lab[use_tr])
    out = {}
    for tname, (pc, nc) in tests.items():
        m = is_test & np.isin(cond, pc + nc); y = np.isin(cond[m], pc).astype(int); sc = predict(X[m])
        keys = [f"{c}|{p}" for c, p in zip(cond[m], prompt[m])]
        tauc, tacc = text_level(sc, y, keys)
        out[tname] = {"token_auc": auc(sc, y), "text_auc": tauc, "text_acc": tacc}
    return out

FEATS = ["mid1", "mid2", "mid3", "mid4", "end4"]
UNSTEERED = ["natural", "prompt_neg", "prompt_pos"]
TESTS_SENT = {"held-out sentiment-steered vs unsteered": (["steer_neg", "steer_pos"], UNSTEERED),
              "transfer -> tense-steered vs natural":     (["steer_tense"], ["natural"]),
              "transfer -> animal-steered vs natural":    (["steer_animal"], ["natural"]),
              "same sentiment: steer_pos vs natural":     (["steer_pos"], ["natural"]),
              "same sentiment: steer_neg vs prompt_neg":  (["steer_neg"], ["prompt_neg"])}
results = {}
print("E1. Probe trained: sentiment-steered vs unsteered (natural + tone-prompted). Text-level AUC on held-out prompts.")
print(f"{'test':<46}" + "".join(f"{f:>8}" for f in FEATS) + "   | after projecting out concept dirs:" + "".join(f"{f:>7}" for f in FEATS[:4]))
for tname in TESTS_SENT:
    row = f"{tname:<46}"
    for f in FEATS:
        r = results.setdefault(("E1", f), run("E1", f, ["steer_neg", "steer_pos"], UNSTEERED, TESTS_SENT))
        row += f"{r[tname]['text_auc']:>8.2f}"
    row += "   |"
    for f in FEATS[:4]:
        r = results.setdefault(("E1d", f), run("E1d", f, ["steer_neg", "steer_pos"], UNSTEERED, TESTS_SENT, debias=True))
        row += f"{r[tname]['text_auc']:>7.2f}"
    print(row)

print("\nE2. Leave-one-concept-out: train steered-vs-unsteered on two concepts, test on the third (text-level AUC, held-out prompts).")
STEER = {"sentiment": ["steer_neg", "steer_pos"], "tense": ["steer_tense"], "animal": ["steer_animal"]}
print(f"{'held-out concept':<20}" + "".join(f"{f:>8}" for f in FEATS) + "   | debiased:" + "".join(f"{f:>7}" for f in FEATS[:4]))
for hold in STEER:
    tr = sum((v for k, v in STEER.items() if k != hold), [])
    tests = {hold: (STEER[hold], UNSTEERED)}
    row = f"{hold:<20}"
    for f in FEATS: row += f"{run('E2', f, tr, UNSTEERED, tests)[hold]['text_auc']:>8.2f}"
    row += "   |"
    for f in FEATS[:4]: row += f"{run('E2d', f, tr, UNSTEERED, tests, debias=True)[hold]['text_auc']:>7.2f}"
    print(row)

print("\nE3. Sentiment confound check (mid4): a SENTIMENT probe (prompt_pos vs prompt_neg) applied to the steering tests.")
r = run("E3", "mid4", ["prompt_pos"], ["prompt_neg"], {"steer_pos vs steer_neg (should be high: content)": (["steer_pos"], ["steer_neg"]),
                                                       "steered vs unsteered (E1 labels)": (["steer_neg", "steer_pos"], UNSTEERED),
                                                       "tense-steered vs natural": (["steer_tense"], ["natural"])})
for k, v in r.items(): print(f"  {k:<50} text AUC {v['text_auc']:.2f}")

print("\nE4. The model's own surprise as a detector (text-level AUC from mean per-token metric, held-out prompts):")
tt = df[is_test]
for tname, (pc, nc) in {"steer_pos vs natural": (["steer_pos"], ["natural"]), "steer_neg vs prompt_neg": (["steer_neg"], ["prompt_neg"]),
                        "tense-steered vs natural": (["steer_tense"], ["natural"]), "animal-steered vs natural": (["steer_animal"], ["natural"])}.items():
    m = tt["cond"].isin(pc + nc); g = tt[m].groupby(["cond", "prompt"])[["nll", "ent", "kl14", "kl34", "gate4"]].mean()
    y = np.isin([c for c, _ in g.index], pc).astype(int)
    print(f"  {tname:<28}" + "".join(f"  {col} {auc(g[col].values, y):.2f}" for col in g.columns))

json.dump({"E1": {f: results[("E1", f)] for f in FEATS}, "E1_debiased": {f: results[("E1d", f)] for f in FEATS[:4]},
           "per_condition_means": df.groupby("cond")[["nll", "ent", "kl14", "kl34", "gate1", "gate4"]].mean().to_dict(),
           "n_tokens": int(len(df)), "n_texts": len(full), "test_prompts": sorted(TEST_PROMPTS)},
          open("/content/steer_detect_results.json", "w"), indent=1)
print("\nsaved /content/steer_detect_results.json")

E1. Probe trained: sentiment-steered vs unsteered (natural + tone-prompted). Text-level AUC on held-out prompts.
test                                              mid1    mid2    mid3    mid4    end4   | after projecting out concept dirs:   mid1   mid2   mid3   mid4
held-out sentiment-steered vs unsteered           0.79    0.78    0.80    0.81    0.83   |   0.80   0.79   0.80   0.81
transfer -> tense-steered vs natural              0.49    0.49    0.47    0.47    0.57   |   0.50   0.51   0.49   0.44
transfer -> animal-steered vs natural             0.64    0.67    0.67    0.67    0.65   |   0.64   0.67   0.67   0.67
same sentiment: steer_pos vs natural              0.67    0.67    0.71    0.69    0.80   |   0.70   0.67   0.72   0.74
same sentiment: steer_neg vs prompt_neg           0.89    0.83    0.84    0.83    0.84   |   0.88   0.86   0.83   0.82

E2. Leave-one-concept-out: train steered-vs-unsteered on two concepts, test on the third (text-level AUC, held-out prompts).
held-out con